# Tuning - XGBoost Regressor para limiar de reposicao

Este notebook desenvolve o XGBoost Regressor, recomendado como segunda opcao para o limiar de reposicao por combinar boa capacidade nao linear com regularizacao forte. Nenhum `.pkl` e salvo localmente; todos os artefatos seguem para o MLflow.


## Estrategia

Usamos `TimeSeriesSplit` pelo mesmo motivo do Random Forest: o problema e temporal e nao deve misturar futuro no treino de folds anteriores. `Stratified K-Fold` nao se aplica ao alvo continuo, `Leave-One-Out` e caro e instavel, e holdout isolado nao mede variancia entre folds.

O tuning usa `RandomizedSearchCV`, mais eficiente que Grid Search para XGBoost porque o espaco inclui profundidade, taxa de aprendizado, subsampling e regularizacoes L1/L2.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "regressor_tuning_common.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
else:
    sys.path.insert(0, str((Path("data") / "notebooks").resolve()))

from regressor_tuning_common import (
    RegressorTuningConfig,
    WEIGHT_PROFILE_DESCRIPTIONS,
    run_regressor_tuning,
)

from xgboost import XGBRegressor


In [ ]:
XGB_PARAM_DISTRIBUTIONS = {
    "model__n_estimators": [150, 250, 400, 600],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.12],
    "model__subsample": [0.70, 0.85, 1.0],
    "model__colsample_bytree": [0.70, 0.85, 1.0],
    "model__min_child_weight": [1, 3, 5, 8],
    "model__gamma": [0.0, 0.05, 0.10, 0.30],
    "model__reg_alpha": [0.0, 0.01, 0.10, 1.0],
    "model__reg_lambda": [0.50, 1.0, 2.0, 5.0],
}

def xgb_model_factory() -> XGBRegressor:
    return XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

def xgb_baseline_factory() -> XGBRegressor:
    return XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
    )

config = RegressorTuningConfig(
    notebook_id="06_xgboost_regressor_threshold_tuning",
    family="Arvores",
    model_name="XGBoost Regressor",
    model_factory=xgb_model_factory,
    baseline_factory=xgb_baseline_factory,
    param_distributions=XGB_PARAM_DISTRIBUTIONS,
    n_iter=20,
    cv_splits=4,
    random_state=42,
)

search_space = pd.DataFrame(
    [{"parametro": key, "valores": values} for key, values in XGB_PARAM_DISTRIBUTIONS.items()]
)
weight_profiles = pd.DataFrame(
    [{"perfil": key, "descricao": value} for key, value in WEIGHT_PROFILE_DESCRIPTIONS.items()]
)

display(search_space)
display(weight_profiles)


## Execucao do tuning

Esta etapa registra no MLflow: baseline, candidatos de Random Search, resultados por fold, melhor combinacao de hiperparametros, modelo campeao, predicoes e graficos.


In [ ]:
results = run_regressor_tuning(config)


## Resultados quantitativos

As tabelas abaixo permitem comparar baseline, top candidatos, variancia dos folds e performance final no holdout de teste.


In [ ]:
display(results["baseline_summary"])
display(results["candidate_results"].head(15))
display(results["best_fold_metrics"])
display(results["final_metrics"].T)
print("Melhor perfil de peso:", results["best_weight_profile"])
print("Melhores hiperparametros:", results["best_params"])
print("Diagnostico:", results["fit_diagnosis"])


## Visualizacoes de ajuste

A curva de aprendizado compara erro de treino e validacao conforme adicionamos dados. A analise de residuos ajuda a encontrar vieses e erros extremos.


In [ ]:
results["figures"]["learning_curve"]


In [ ]:
results["figures"]["residual_analysis"]


## MLflow

No MLflow, procure pelo experimento `saltim_two_stage_06_xgboost_regressor_threshold_tuning_threshold_regression`. O run `champion` contem o modelo registrado; os runs de candidatos mostram os diferentes parametros e perfis de peso avaliados.
